# Validate Text-Guided Cross-Attention Fusion Model
Notebook này được sử dụng để chạy validation trên mô hình dạng multimodal, kết hợp giữa `ConvNeXtTextGuidedCBAMEncoder` và `MultimodalCrossAttnClassifier`.

**Nguồn dữ liệu:**
- Image: `data/AIDG/dataset_PlantDoc/images/val`
- Text (Captions): `data/AIDG/captions_LLaVA/val`


In [6]:
import os
import json
import sys
import re
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.notebook import tqdm
import open_clip
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Di chuyển context ra ngoài thư mục gốc của project
PROJECT_ROOT = Path("../../").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.models.backbones.vision.convnext_text_guided_cbam_encoder import ConvNeXtTextGuidedCBAMEncoder
from src.models.multimodal.classifier_cross_attn import MultimodalCrossAttnClassifier

In [7]:
# =========================
# 1. CẤU HÌNH ĐƯỜNG DẪN
# =========================
VAL_IMAGE_ROOT = PROJECT_ROOT / "data" / "AIDG" / "dataset_PlantDoc" / "images" / "val"
VAL_CAPTION_ROOT = PROJECT_ROOT / "data" / "AIDG" / "captions_LLaVA" / "val" 
CKPT_PATH = PROJECT_ROOT / "archive" / "text-guided-cross-attn-fusion" / "best_model.pt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
IMG_SIZE = 224
NUM_WORKERS = 4

print("Image Dir:", VAL_IMAGE_ROOT)
print("Caption Dir:", VAL_CAPTION_ROOT)

Image Dir: /media/data3/users/luongdth/MulCo-PlantNet/data/AIDG/dataset_PlantDoc/images/val
Caption Dir: /media/data3/users/luongdth/MulCo-PlantNet/data/AIDG/captions_LLaVA/val


In [8]:
# =========================
# 2. DATASET VÀ DATALOADER
# =========================
def normalize_caption_for_clip(text: str) -> str:
    text = (text or "").strip()
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    for i in range(1, 8):
        text = re.sub(rf"(?i)\bstep\s*{i}\s*:", f"Step {i}:", text)
        text = re.sub(rf"(?i)\bstep{i}\b", f"Step {i}", text)
    text = " ".join(line.strip() for line in text.split("\n") if line.strip())
    text = re.sub(r"\s+", " ", text).strip()
    return text

class MultimodalValDataset(Dataset):
    def __init__(self, image_dir, caption_dir, transform=None):
        self.image_dir = Path(image_dir)
        self.caption_dir = Path(caption_dir)
        self.transform = transform
        
        self.samples = []
        self.classes = sorted([d.name for d in self.image_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        for cls_name in self.classes:
            cap_file = self.caption_dir / f"{cls_name}.json"
            captions_dict = {}
            if cap_file.exists():
                with open(cap_file, "r", encoding="utf-8") as f:
                    captions_dict = json.load(f)
            else:
                print(f"[Warning] No caption file found for {cls_name} at {cap_file}")
                
            cls_dir = self.image_dir / cls_name
            if not cls_dir.exists():
                continue
                
            for img_path in cls_dir.glob("*.*"):
                if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
                    continue
                    
                img_key = img_path.name
                raw_caption = captions_dict.get(img_key, {}).get("text", "")
                caption = normalize_caption_for_clip(raw_caption)
                
                if not caption:
                    # Cung cấp một caption mặc định nếu thiếu
                    caption = f"A picture of a {cls_name} leaf."
                    
                self.samples.append({
                    "image_path": img_path,
                    "class_name": cls_name,
                    "label": self.class_to_idx[cls_name],
                    "caption": caption
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample["image_path"]).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        return {
            "image": image,
            "caption": sample["caption"],
            "label": sample["label"],
            "class_name": sample["class_name"]
        }

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_dataset = MultimodalValDataset(VAL_IMAGE_ROOT, VAL_CAPTION_ROOT, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Tổng số mẫu validation: {len(val_dataset)}")
print(f"Số lượng class: {len(val_dataset.classes)}")

Tổng số mẫu validation: 635
Số lượng class: 28


In [9]:
# =========================
# 3. KHỞI TẠO PIPELINE MODEL
# =========================
class TextGuidedCrossAttnPipeline(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # 1. Vision backbone guided by text
        self.image_encoder = ConvNeXtTextGuidedCBAMEncoder(text_dim=768)
        
        # 2. Classifier sử dụng Cross-Attention Fusion
        self.fusion = MultimodalCrossAttnClassifier(
            image_input_dim=1024, 
            text_input_dim=768, 
            num_classes=num_classes
        )
        
    def forward(self, image, text_feat):
        # Trích xuất Image Feature có sự điều hướng từ Text Feature
        image_feat = self.image_encoder(image, text_feat)
        
        # Cross-Attention Fusion -> Logits
        logits = self.fusion(image_feat, text_feat)
        return logits

num_classes = len(val_dataset.classes)
model = TextGuidedCrossAttnPipeline(num_classes=num_classes).to(DEVICE)

# Load Pre-trained weights
if not CKPT_PATH.exists():
    raise FileNotFoundError(f"❌ Không tìm thấy checkpoint tại {CKPT_PATH}")

ckpt = torch.load(CKPT_PATH, map_location="cpu")
state_dict = ckpt.get("state_dict", ckpt)
state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
if missing_keys:
    print(f"⚠️ Cảnh báo - Missing keys (weights không khớp): {len(missing_keys)} keys bị thiếu.")
    print("Ví dụ vài key thiếu:", missing_keys[:5])
else:
    print("✅ Tải checkpoint thành công và khớp hoàn toàn!")

model.eval()

# Load mô hình CLIP (đóng băng weights)
print("Đang tải CLIP ViT-L/14 model...")
clip_model, _, _ = open_clip.create_model_and_transforms("ViT-L-14", pretrained="openai")
clip_model = clip_model.to(DEVICE).eval()
clip_tokenizer = open_clip.get_tokenizer("ViT-L-14")

FileNotFoundError: ❌ Không tìm thấy checkpoint tại /media/data3/users/luongdth/MulCo-PlantNet/archive/text-guided-cross-attn-fusion/best_model.pt

In [ ]:
# =========================
# 4. CHẠY VALIDATION LOOP
# =========================
correct = 0
total = 0
running_loss = 0.0
criterion = nn.CrossEntropyLoss()

all_preds = []
all_labels = []

print("Bắt đầu Validation...")
with torch.no_grad():
    for batch in tqdm(val_loader, desc="Validating"):
        images = batch["image"].to(DEVICE)
        captions = batch["caption"]
        labels = batch["label"].to(DEVICE)
        
        # 1. Trích xuất text features qua CLIP
        tokens = clip_tokenizer(captions).to(DEVICE)
        text_feat = clip_model.encode_text(tokens)
        text_feat = F.normalize(text_feat, dim=-1) # Dữ liệu văn bản normalize chuẩn L2 [B, 768]
        
        # 2. Truyền sang Pipeline (Image Feature sinh ra dựa vào text -> Fusion -> Classify)
        logits = model(images, text_feat)
        
        # 3. Tính Metrics
        loss = criterion(logits, labels)
        running_loss += loss.item() * images.size(0)
        
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

epoch_loss = running_loss / total
epoch_acc = correct / total

print(f"\n🎯 Validation Loss: {epoch_loss:.4f}")
print(f"🎯 Validation Accuracy: {epoch_acc * 100:.2f}%")

In [ ]:
# =========================
# 5. IN RA BÁO CÁO & MA TRẬN NHẦM LẪN
# =========================
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=val_dataset.classes))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(16, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=val_dataset.classes, yticklabels=val_dataset.classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Validation Set")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()